# Simultaneous games and Nash equilibrium

_Artificial Intelligence — more_

**Both move at once. A stable outcome is one where no one wishes they'd chosen differently.**

In a simultaneous game, players choose at the same time, without seeing each other's move. A payoff matrix lists everyone's reward for each combination.

     A Nash equilibrium is a choice for each player where no single player can do better by switching alone.

---

This notebook is a practice scaffold for the **Simultaneous games and Nash equilibrium** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

### Step 1 — Write down the payoff matricesThis is the Prisoner's Dilemma. Each player picks one of two moves, Cooperate or Defect, at the same time. `A[r, c]` is player A's payoff and `B[r, c]` is player B's payoff when A plays row `r` and B plays column `c`. Higher payoffs are better (these are negative because they represent years in prison).

In [ ]:
import numpy as np

# rows = A's move {0=Cooperate, 1=Defect}; cols = B's move. Higher payoff is better.
A = np.array([[-1, -3],                    # A's payoff
              [ 0, -2]])
B = np.array([[-1,  0],                    # B's payoff
              [-3, -2]])
names = ["Cooperate", "Defect"]

### Step 2 — Search for Nash equilibriaAn outcome `(r, c)` is a **Nash equilibrium** when neither player can improve by changing their move alone. For player A, that means `A[r, c]` is the best payoff available in column `c` (A can only switch rows). For player B, `B[r, c]` must be the best in row `r` (B can only switch columns). We check every cell and print the ones where both conditions hold.

In [ ]:
print("Nash equilibria (each player best-responding):")
for r in range(2):
    for c in range(2):
        a_best = A[r, c] == A[:, c].max()   # A can't improve by changing row
        b_best = B[r, c] == B[r, :].max()   # B can't improve by changing col
        if a_best and b_best:
            print(f"  (A:{names[r]}, B:{names[c]}) payoffs ({A[r,c]},{B[r,c]})")

### Step 3 — Check for a dominant strategyA move is **dominant** for a player if it beats their alternative no matter what the opponent does. Comparing A's Defect row against A's Cooperate row entry by entry, Defect wins in every column — so Defect strictly dominates for A. This is why both players defect even though mutual cooperation would have been better for both.

In [ ]:
# Defect dominates for A if A's Defect row beats its Cooperate row everywhere.
defect_dominant = (A[1] > A[0]).all()
print("Defect dominant for A?", bool(defect_dominant))

## Visualize the data & results

_Soccer penalty kicks: what mix of Left/Right should a right-footed kicker use, and what is the scoring rate at equilibrium?_

### Step 1 — Build the penalty-kick payoff matrixThis is a zero-sum game between a kicker and a goalie. Each entry of `K` is the kicker's scoring chance (%) for a combination of kick direction (row) and goalie dive (column). The kicker is right-footed and shoots better to their Left, so the matrix is asymmetric — the goalie wants to minimise exactly what the kicker wants to maximise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Penalty-kick zero-sum game. Entry = kicker scoring chance (%).
# Rows = kicker direction, cols = goalie dive. Right-footed kicker is stronger
# shooting to their Left, so the matrix is asymmetric.
K = np.array([[58.0, 95.0],                # kicker Left:  goalie Left=58, goalie Right=95
              [93.0, 70.0]])               # kicker Right: goalie Left=93, goalie Right=70

### Step 2 — Solve the mixed-strategy equilibriumNeither pure direction is safe here, so the kicker should **randomise**. For a 2x2 zero-sum game there is a closed-form mix: the kicker picks the probability `q` of aiming Left that makes the goalie indifferent between diving Left or Right. Plugging that `q` back in gives the game's **value** — the kicker's scoring rate at equilibrium.

In [ ]:
# Name the four matrix entries for the closed-form solution.
a, b, c, d = K[0, 0], K[0, 1], K[1, 0], K[1, 1]

# Kicker's equilibrium probability of aiming Left.
q = (d - c) / (a - b - c + d)              # 0.383

# Scoring chance at equilibrium (the value of the game).
value = a * q + c * (1 - q)                # 79.6% scoring chance
print("aim Left prob=", round(q, 3), " game value=", round(value, 1), "%")

### Step 3 — Visualise the payoff matrixFinally we draw the scoring-chance matrix as a heatmap, annotating each cell with its value. Reading it alongside the equilibrium mix above shows why the kicker must randomise: no single row is best against both goalie dives, so mixing is the only stable strategy.

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(K, cmap="magma")

# Annotate each cell with its scoring chance.
for r in range(2):
    for c2 in range(2):
        ax.text(c2, r, int(K[r, c2]), ha="center", va="center", color="w")

ax.set_xticks([0, 1])
ax.set_xticklabels(["Goalie dives Left", "Goalie dives Right"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["Kicker aims Left", "Kicker aims Right"])
ax.set_title("Kicker scoring chance (%) by kicker direction vs goalie dive")
fig.colorbar(im, ax=ax)
plt.show()